In [ ]:
import sympy as sp
import networkx as nx
import matplotlib.pyplot as plt

def EjeTab():
    from sympy import Rational, Matrix, symbols, solve, N
    import ast
    
    print("\n--- Ejemplo: Tablero (naranja,azul,verde,rojo) ---")
    print("Matriz de transición P (filas: actual, columnas: sig):")
    
    P = Matrix([[Rational(1,4), Rational(1,2), Rational(1,4), 0],
        [0, Rational(1,4), Rational(1,2), Rational(1,4)],
        [Rational(1,4), 0, Rational(1,4), Rational(1,2)],
        [Rational(1,2), Rational(1,4), 0, Rational(1,4)]])
    sp.pprint(P)
    print("(doble estocástica: filas y columnas suman 1)")
    
    print("\nVector inicial (lista 4 nums, ej. [0,0,0,1]):")
    while True:
        try:
            ent = input("v0 = ")
            lst = ast.literal_eval(ent)
            if not isinstance(lst, list) or len(lst) != 4:
                raise ValueError("Se necesita lista de 4 elementos.")
            v0 = Matrix([Rational(x) for x in lst])
            break
        except Exception as e:
            print(f"Error: {e}. Intente otra vez.")
    
    print("\nOpciones:")
    print("a) Iterar n pasos")
    print("b) Distribución estacionaria (analítica)")
    print("c) Iterar hasta convergencia")
    opc = input("Elija una opcion ").strip().lower()
    
    if opc=='a':
        try:
            n=int(input("Número de pasos: "))
            if n<0: raise ValueError
        except:
            print("Número inválido.")
            return
        print("\nEvolución:")
        v=v0
        print(f"paso 0: {v.T}  suma={sum(v)}")
        for k in range(1, n+1):
            v=P*v
            print(f"paso {k}: {v.T}  suma={sum(v)}")
        print(f"\nResultado final:")
        sp.pprint(v)
    
    elif opc == 'b':
        print("\nDistribución estacionaria π tal que πP=π, suma=1.")
        p = symbols('p0 p1 p2 p3')
        ecs = []
        for i in range(4):
            ec = 0
            for j in range(4):
                ec+=p[j]*(P[j,i]-(1 if i==j else 0))
            ecs.append(ec)
        ecs.append(p[0]+p[1]+p[2]+p[3]-1)
        sol=solve(ecs, p)
        pi=Matrix([sol[pi] for pi in p])
        print("π =")
        sp.pprint(pi)
        print("\nVerificación πP:")
        sp.pprint(pi.T * P)
    
    elif opc == 'c':
        print("\nIterando hasta convergencia:")
        v=v0
        paso=0
        dif = float('inf')
        tol = 1e-10
        maxp = 10000
        while dif > tol and paso < maxp:
            vn = P * v
            dif_raw = max(abs(vn[i] - v[i]) for i in range(4))
            dif = float(dif_raw)
            v = vn
            paso += 1
            if paso % 10 == 0 or paso < 5:
                print(f"paso {paso}: {v.T}  dif={N(dif_raw):.2e}")
        print(f"\nConvergido en {paso} pasos.")
        print("Distribución límite:")
        sp.pprint(v)
        print("Teórica: [1/4, 1/4, 1/4, 1/4]")
    
        import matplotlib.pyplot as plt
        nodos = ['Naranja', 'Azul', 'Verde', 'Rojo']
        G = nx.DiGraph()
        G.add_nodes_from(nodos)
        for i in range(4):
            for j in range(4):
                w = P[i,j]
                if w != 0:
                    G.add_edge(nodos[i], nodos[j], weight=float(w))
        pos = nx.spring_layout(G)
        nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=500)
        nx.draw_networkx_labels(G, pos, font_size=12)
        nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, edge_color='gray')
        etiq = {(u,v): f"{d['weight']:.2f}" for u,v,d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=etiq, font_size=10)
        plt.title("Transición del tablero")
        plt.axis('off')
        plt.show()

if __name__ == "__main__":
    EjeTab()


--- Ejemplo: Tablero (naranja,azul,verde,rojo) ---
Matriz de transición P (filas: actual, columnas: sig):
⎡1/4  1/2  1/4   0 ⎤
⎢                  ⎥
⎢ 0   1/4  1/2  1/4⎥
⎢                  ⎥
⎢1/4   0   1/4  1/2⎥
⎢                  ⎥
⎣1/2  1/4   0   1/4⎦
(doble estocástica: filas y columnas suman 1)

Vector inicial (lista 4 nums, ej. [0,0,0,1]):
